### Load The PDF Document

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("./youtube_guidelines.pdf")
documents = loader.load()

print(f"Loaded {len(documents)} documents.")

/Users/sujith/Documents/Agentic_AI_Learning/agent_ai/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 11 documents.


### Split the Documents into Chunks

In [2]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks= text_splitter.split_documents(documents)

### Create Embeddings

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9032.31it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Create the Vector Database and set up retriever

In [5]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents = chunks, 
                embedding = embeddings, 
                persist_directory="./chroma_db",
                collection_name="resume_collection")

print("Vector store created.")

Vector store created.


In [6]:
retriever = vectorstore.as_retriever(search_kwargs={"k":3})

### Connect to the Local LLM

In [7]:
from langchain_ollama import OllamaLLM
llm= OllamaLLM(model="llama3")

### Defining the Chat Prompt Template

In [8]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
    You are a Helpful AI Assistant.
    
    Use the following retrieved context to answer the question.
    
    Context: {context}
    
    Question: {question}
    
    Answer consisely and accurately based on the provided context in no more than 3 sentences.
    """
    
)

### Build the RAG Pipeline

In [9]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "/n/n".join(doc.page_content for doc in docs)

rag_chain = ({
    "context": retriever | format_docs,
    "question": RunnablePassthrough()
    }
    | prompt | llm | StrOutputParser())

In [10]:
while True:
    question= input("Ask a Question: ")
    if question.lower() in ("exit", "quit"):
        print("Exiting the RAG system. Goodbye!")
        break
    response = rag_chain.invoke(question)
    print(f"\nAnswer: {response}")


Answer: According to the provided context, creators should use royalty-free music from YouTube Audio Library, Epidemic Sound, or Artlist for background music. Using even 3 seconds of a copyrighted song without a licence can result in a Content ID claim on the entire video. It is important to note that creators should not use copyrighted music, as it may lead to a Content ID claim.

Answer: According to the provided context, the information about background music can be found in Section 5 — Important Note. Specifically, it is mentioned that creators should use royalty-free music from YouTube Audio Library, Epidemic Sound, or Artlist for background music, and that using even 3 seconds of a copyrighted song without a licence can result in a Content ID claim on the entire video.
Exiting the RAG system. Goodbye!
